# Неделя 3 — Признаки и CatBoost

Задание: `docs/week3_baseline.md`

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from src.validation import evaluate, time_split

dataset = pd.read_parquet('../data/processed/dataset.parquet')
CAT_COLS = ['segment', 'product', 'region']
NUM_COLS = [c for c in dataset.columns
            if c not in CAT_COLS + ['client_id', 'snapshot_date', 'target']]
print(len(NUM_COLS), 'числовых признаков')

18 числовых признаков


## 1. Построение признаков по группам
Признаки строит `make_features()` в `src/features.py` (вызывается внутри
`build_snapshot`, в датасет попадают при сборке на неделе 2). Группы:
- платежи: revenue_mean/min/max/last, revenue_last_to_mean
- динамика: revenue_trend, revenue_volatility, revenue_declining_months
- объём отношений: n_sim_mean_6m, n_sim_last, tenure_months
- сигналы боли: tickets_sum/mean_6m, debt_max/mean/last
- использование: traffic_mean/last
- категориальные: segment, product, region

In [2]:
print(NUM_COLS)

['tenure_months', 'revenue_mean_6m', 'revenue_min_6m', 'revenue_max_6m', 'revenue_last', 'traffic_mean_6m', 'traffic_last', 'tickets_sum_6m', 'tickets_mean_6m', 'debt_max_6m', 'debt_mean_6m', 'debt_last', 'n_sim_mean_6m', 'n_sim_last', 'revenue_last_to_mean', 'revenue_trend', 'revenue_volatility', 'revenue_declining_months']


## 2. Обучение CatBoost


In [3]:
train, val, test = time_split(dataset, ['2024-07', '2024-10'], ['2025-01'], ['2025-10'])
X_train, y_train = train[NUM_COLS + CAT_COLS], train['target']
X_val, y_val = val[NUM_COLS + CAT_COLS], val['target']
X_test, y_test = test[NUM_COLS + CAT_COLS], test['target']

model_cb = CatBoostClassifier(
    iterations=1000,
    early_stopping_rounds=100,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=False,
)
model_cb.fit(X_train, y_train, cat_features=CAT_COLS,
             eval_set=(X_val, y_val), use_best_model=True)
print('best_iteration:', model_cb.best_iteration_)

score_cb = model_cb.predict_proba(X_test)[:, 1]
m_cb = evaluate(y_test, score_cb, 'CatBoost')

best_iteration: 135
--- CatBoost ---
ROC-AUC: 0.8435
PR-AUC: 0.5322
Precision@10%: 0.5263
Lift@10%: 6.48x
Доля оттока: 0.0812


## 3. Итоговая таблица метрик: правило vs логрегрессия vs CatBoost

In [4]:
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

m_rule = evaluate(y_test, -X_test['revenue_last_to_mean'], 'Правило')

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('log', FunctionTransformer(signed_log1p, validate=False)),
        ('scale', RobustScaler()),
    ]), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
])
model_lr = Pipeline([
    ('pre', preprocess),
    ('clf', LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced')),
])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    model_lr.fit(X_train, y_train)
score_lr = model_lr.predict_proba(X_test)[:, 1]
m_lr = evaluate(y_test, score_lr, 'Логрегрессия')

final = pd.DataFrame([m_rule, m_lr, m_cb]).set_index('label')
print(final[['roc_auc', 'pr_auc', 'precision_at_10', 'lift_at_10']].round(4))

--- Правило ---
ROC-AUC: 0.7934
PR-AUC: 0.3597
Precision@10%: 0.4596
Lift@10%: 5.66x
Доля оттока: 0.0812
--- Логрегрессия ---
ROC-AUC: 0.8270
PR-AUC: 0.4813
Precision@10%: 0.4898
Lift@10%: 6.03x
Доля оттока: 0.0812
              roc_auc  pr_auc  precision_at_10  lift_at_10
label                                                     
Правило        0.7934  0.3597           0.4596      5.6583
Логрегрессия   0.8270  0.4813           0.4898      6.0298
CatBoost       0.8435  0.5322           0.5263      6.4797


/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 4. Вывод на языке бизнеса


In [5]:
lift = m_cb['lift_at_10']
capture = lift * 0.10
print(f"В топ-10% клиентов по риску попадает {capture:.0%} всех будущих уходов, "
      f"lift = {lift:.1f}x (ROC-AUC = {m_cb['roc_auc']:.3f})")

В топ-10% клиентов по риску попадает 65% всех будущих уходов, lift = 6.5x (ROC-AUC = 0.843)


### Почему AUC больше не 0.946
В первой версии не было зазора между окном наблюдения и окном прогноза:
коллапс выручки (начинается за 4-5 месяцев до ухода — см. event study недели 1)
попадал прямо в признаки, и модель не предсказывала будущее, а читала настоящее.
После вставки зазора в 2 месяца и удаления пересечения train/test (выкинуты
снапшоты 2025-04 и 2025-07) задача вернулась к честной сложности: AUC правила
упал с 0.907 до 0.79, CatBoost — около 0.85. Это правдоподобный диапазон 0.7-0.9
из задания.